# 7. Indeks Mie Ayam: Bagian 2 (Biaya Semangkuk)

Bagian 1 sudah mengenalkan data. Bagian 2 menyiapkan data dan menghitung biaya bahan semangkuk mie ayam per bulan, lalu **mencocokkannya dengan Notebook 2** (Rp 2.763 di Jan 2021 dan Rp 3.613 di Sep 2026). Indeks (Jan 2021 = 100) dan dashboard dikerjakan di bagian berikutnya.

## Resep asumsi per porsi (sama dengan Notebook 2)
- Ayam 60 g
- Bawang merah 10 g
- Bawang putih 5 g
- Minyak goreng 20 ml

## Langkah
1. Ubah `bulan` jadi tanggal, buang Okt 2026, ubah data jadi format lebar, isi bulan kosong.
2. Cek satuan harga.
3. Hitung biaya dengan tiga cara memakai dua merek minyak, lalu cocokkan dengan Notebook 2.

In [ ]:
import pandas as pd

path = '/kaggle/input/datasets/samuelgeraldo/bedah-mie-ayam-pakai-data-bukan-sumpit/harga_pangan_bersih.csv'
df = pd.read_csv(path)

# bulan: teks -> tanggal, lalu buang Okt 2026 (belum utuh)
df['bulan'] = pd.to_datetime(df['bulan'], format='%Y-%m')
df = df[df['bulan'] < '2026-10-01']

# format panjang -> lebar (satu kolom per komoditas)
wide = df.pivot(index='bulan', columns='komoditas', values='harga_rp')
wide = wide.rename(columns={
    'Bawang Merah Ukuran Sedang': 'bawang_merah',
    'Bawang Putih Ukuran Sedang': 'bawang_putih',
    'Daging Ayam Ras Segar': 'ayam',
    'Minyak Goreng Kemasan Bermerk 1': 'minyak1',
    'Minyak Goreng Kemasan Bermerk 2': 'minyak2',
})

# isi bulan kosong: Mei-Jun 2022 dengan garis lurus, Jan 2021 disalin dari Feb 2021
wide = wide.interpolate(method='linear').bfill()

print(wide.shape, 'sel kosong:', wide.isna().sum().sum())
wide.head()

**Hasilnya:** (isi setelah dijalankan: harusnya 69 baris x 5 kolom dan 0 sel kosong.)

## Cek satuan harga
Rumus butuh tahu apakah harga di data per kg (ayam, bawang) dan per liter (minyak). Kita lihat bulan pertama dan terakhir.

In [ ]:
wide.iloc[[0, -1]].round(0)

**Hasilnya:** (isi setelah dijalankan: ayam dan bawang puluhan ribu = per kg, minyak belasan ribu = per liter.)

## Hitung biaya semangkuk
Resep hanya punya satu bahan minyak (20 ml), tapi data punya dua merek. Kita coba tiga cara dan lihat mana yang cocok dengan Notebook 2:
- rata-rata dua merek
- hanya merek 1
- hanya merek 2

Gram dibagi 1000 jadi kg, ml dibagi 1000 jadi liter.

In [ ]:
def biaya(minyak):
    return (wide['ayam'] * 0.06
            + wide['bawang_merah'] * 0.01
            + wide['bawang_putih'] * 0.005
            + minyak * 0.02)

opsi = pd.DataFrame({
    'rata2_dua_merek': biaya(wide[['minyak1', 'minyak2']].mean(axis=1)),
    'merek1_saja': biaya(wide['minyak1']),
    'merek2_saja': biaya(wide['minyak2']),
})

print('Target Notebook 2: Jan 2021 = 2763, Sep 2026 = 3613')
opsi.loc[['2021-01-01', '2026-09-01']].round(0)

**Hasilnya:** (isi setelah dijalankan: kolom mana yang cocok dengan 2763 dan 3613?)

## Kesimpulan bagian 2
(isi setelah dijalankan: cara memakai dua merek minyak yang sama dengan Notebook 2, dan apakah biaya semangkuk sudah cocok.)

Bagian berikutnya: ubah biaya jadi indeks (Jan 2021 = 100) dan siapkan data untuk dashboard.